# Build 1 · Lakebase Search — execution proof
The app's product recommendations retrieve from the **Build-1 Lakebase Search index** (native hybrid vector + BM25 over `app.products`) — **not a separate vector store**. Retrieval never leaves Lakebase. This notebook runs the exact retrieval the app uses (`server/main.py::rm_recommendations`) and shows the output.

In [1]:
import os, json, subprocess, psycopg
from databricks.sdk import WorkspaceClient
os.environ['DATABRICKS_CONFIG_PROFILE']='fevm-ts'
def cj(a): return json.loads(subprocess.run(a,capture_output=True,text=True).stdout)
B='projects/meridian-bank/branches/production'
host=cj(['databricks','postgres','list-endpoints',B,'-p','fevm-ts','-o','json'])[0]['status']['hosts']['host']
tok=cj(['databricks','postgres','generate-database-credential',B+'/endpoints/primary','-p','fevm-ts','-o','json'])['token']
cx=psycopg.connect(host=host,port=5432,dbname='databricks_postgres',user='akash.s@databricks.com',password=tok,sslmode='require',autocommit=True)
cur=cx.cursor(); print('Connected to Lakebase:', host)

Connected to Lakebase: ep-gentle-wind-d2ydzmm4.database.us-east-1.cloud.databricks.com


### 1. The Build-1 Lakebase Search index exists on `app.products` (vector + BM25)

In [2]:
cur.execute("select extname from pg_extension where extname in ('lakebase_vector','lakebase_text') order by 1")
print('extensions:', [r[0] for r in cur.fetchall()])
cur.execute("select indexname from pg_indexes where schemaname='app' and tablename='products' and (indexdef ilike '%lakebase_ann%' or indexdef ilike '%lakebase_bm25%') order by 1")
print('search indexes on app.products:', [r[0] for r in cur.fetchall()])

extensions: ['lakebase_text', 'lakebase_vector']


search indexes on app.products: ['products_ann', 'products_bm25']


### 2. Embed the customer query (databricks-gte-large-en) and run the hybrid RRF search

In [3]:
w=WorkspaceClient(profile='fevm-ts')
q='affluent long-tenure customer with a maturing CD; wealth advisory and investment options to deepen the relationship'
emb=w.serving_endpoints.query('databricks-gte-large-en', input=[q])
vec=emb.data[0]['embedding'] if isinstance(emb.data[0],dict) else emb.data[0].embedding
print('query embedding dims:', len(vec))
vlit='['+','.join(str(x) for x in vec)+']'

query embedding dims: 1024


In [4]:
sql='''
WITH vector_ranked AS (
  SELECT product_id, RANK() OVER (ORDER BY dist) rank FROM (
    SELECT product_id, embedding <=> %(qv)s::vector dist
    FROM app.products WHERE embedding IS NOT NULL ORDER BY dist LIMIT 40) v),
keyword_ranked AS (
  SELECT product_id, RANK() OVER (ORDER BY score) rank FROM (
    SELECT product_id, search_tsv <@> to_bm25query(to_tsvector('english', %(qt)s), 'app.products_bm25') score
    FROM app.products ORDER BY score LIMIT 40) k)
SELECT p.product_id, p.product_name, p.product_type,
  round((COALESCE(1.0/(60+v.rank),0)+COALESCE(1.0/(60+k.rank),0))::numeric,6) rrf_score
FROM app.products p LEFT JOIN vector_ranked v USING(product_id) LEFT JOIN keyword_ranked k USING(product_id)
WHERE v.product_id IS NOT NULL OR k.product_id IS NOT NULL
ORDER BY rrf_score DESC, p.product_id LIMIT 5'''
cur.execute(sql, {'qv': vlit, 'qt': q})
print('Top-5 recommendations from the Lakebase Search index (hybrid RRF):')
for r in cur.fetchall(): print(' ', r[0], '|', r[1], '|', r[2], '| rrf=', float(r[3]))

Top-5 recommendations from the Lakebase Search index (hybrid RRF):
  PROD-INV-3001 | Wealth Advisory Account | Advisory | rrf= 0.016393
  PROD-LN-5002 | 30-Year Fixed Mortgage | Mortgage | rrf= 0.016129
  PROD-CRD-4001 | Premier Rewards Credit Card | Card | rrf= 0.015873
  PROD-INV-3002 | Self-Directed Brokerage | Brokerage | rrf= 0.015873
  PROD-DEP-2001 | 18-Month Certificate of Deposit | CD | rrf= 0.015385


**Result:** ranked products come straight from the `lakebase_ann` + `lakebase_bm25` indexes on `app.products` inside Lakebase — hybrid vector + keyword retrieval, fused with Reciprocal Rank Fusion, with **no external/separate vector store**. This is the exact path the app's `GET /api/rm/recommendations/{customer_id}` uses.